# Training Klasifikasi Laporan Kerusakan Fasilitas Kampus
---
**Model:** IndoBERT (indobenchmark/indobert-base-p2)

**Tasks:**
1. Klasifikasi **Kategori** (4 kelas)
2. Klasifikasi **Urgensi** (3 kelas)

**Kaggle Settings:**
- Accelerator: GPU P100 / T4
- Input: dataset_smart_router_final.csv

In [ ]:
# ============================================================
# 0. INSTALL DEPENDENCIES (Kaggle may have these pre-installed)
# ============================================================
!pip install transformers datasets accelerate evaluate -q

In [ ]:
import json
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
)
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# ============================================================
# 1. CONFIGURATION
# ============================================================
DATA_PATH = "/data/dataset_smart_router_final.csv"
OUTPUT_DIR = Path("models")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

MODEL_NAME = "indobenchmark/indobert-base-p2"
MAX_LENGTH = 256
BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 2e-5

print(f"Output dir: {OUTPUT_DIR}")
print(f"Model: {MODEL_NAME}")
print(f"Max length: {MAX_LENGTH}, Batch: {BATCH_SIZE}, Epochs: {EPOCHS}, LR: {LEARNING_RATE}")

In [ ]:
# ============================================================
# 2. LOAD & EXPLORATORY DATA ANALYSIS (EDA)
# ============================================================
df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

In [ ]:
print("=== KATEGORI DISTRIBUTION ===")
print(df['kategori'].value_counts())
print()
print("=== URGENSI DISTRIBUTION ===")
print(df['urgensi'].value_counts())

In [ ]:
# Text cleaning function
def clean_text(text):
    if not isinstance(text, str):
        return ""
    return text.strip().lower()

df["text_clean"] = df["text_input"].apply(clean_text)
df["char_len"] = df["text_clean"].apply(len)
df["word_len"] = df["text_clean"].apply(lambda x: len(x.split()))

print(f"Avg chars: {df['char_len'].mean():.1f} (min={df['char_len'].min()}, max={df['char_len'].max()})")
print(f"Avg words: {df['word_len'].mean():.1f} (min={df['word_len'].min()}, max={df['word_len'].max()})")

In [ ]:
# EDA Visualizations
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

df["kategori"].value_counts().plot(kind="bar", ax=axes[0, 0], title="Kategori Distribution", color="steelblue")
axes[0, 0].tick_params(axis='x', rotation=45)

df["urgensi"].value_counts().plot(kind="bar", ax=axes[0, 1], title="Urgensi Distribution", color="coral")

df.boxplot(column="char_len", by="kategori", ax=axes[0, 2])
axes[0, 2].set_title("Char Length by Kategori")
axes[0, 2].tick_params(axis='x', rotation=45)

df.boxplot(column="char_len", by="urgensi", ax=axes[1, 0])
axes[1, 0].set_title("Char Length by Urgensi")

df.boxplot(column="word_len", by="kategori", ax=axes[1, 1])
axes[1, 1].set_title("Word Length by Kategori")
axes[1, 1].tick_params(axis='x', rotation=45)

df.boxplot(column="word_len", by="urgensi", ax=axes[1, 2])
axes[1, 2].set_title("Word Length by Urgensi")

plt.suptitle("Exploratory Data Analysis")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "eda_plots.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# 3. LABEL ENCODING
# ============================================================
le_kategori = LabelEncoder()
le_urgensi = LabelEncoder()

df["label_kategori"] = le_kategori.fit_transform(df["kategori"])
df["label_urgensi"] = le_urgensi.fit_transform(df["urgensi"])

print("Kategori classes:", le_kategori.classes_)
print("Urgensi classes:", le_urgensi.classes_)
print()
print("Kategori mapping:")
for i, cls in enumerate(le_kategori.classes_):
    print(f"  {i} -> {cls}")
print("\nUrgensi mapping:")
for i, cls in enumerate(le_urgensi.classes_):
    print(f"  {i} -> {cls}")

In [ ]:
# Save label encoders for later inference
with open(OUTPUT_DIR / "label_encoder_kategori.pkl", "wb") as f:
    pickle.dump(le_kategori, f)
with open(OUTPUT_DIR / "label_encoder_urgensi.pkl", "wb") as f:
    pickle.dump(le_urgensi, f)
print("Label encoders saved.")

In [ ]:
# ============================================================
# 4. TRAIN / VALIDATION / TEST SPLIT
# ============================================================
X = df["text_clean"].values
yk = df["label_kategori"].values
yu = df["label_urgensi"].values

# Stratified split: 80% train, 10% val, 10% test
X_temp, X_test, yk_temp, yk_test, yu_temp, yu_test = train_test_split(
    X, yk, yu, test_size=0.1, random_state=42, stratify=yk
)
X_train, X_val, yk_train, yk_val, yu_train, yu_val = train_test_split(
    X_temp, yk_temp, yu_temp, test_size=0.1111, random_state=42, stratify=yk_temp
)

print(f"Train: {len(X_train)} samples")
print(f"Val:   {len(X_val)} samples")
print(f"Test:  {len(X_test)} samples")
print()
print(f"Kategori train distribution: {np.bincount(yk_train)}")
print(f"Urgensi train distribution:  {np.bincount(yu_train)}")

In [ ]:
# ============================================================
# 5. TOKENIZATION & DATASET CREATION
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.save_pretrained(str(OUTPUT_DIR / "tokenizer"))
print(f"Tokenizer loaded. Vocab size: {tokenizer.vocab_size}")

In [ ]:
class TextClassificationDataset(Dataset):
    """PyTorch Dataset for text classification."""
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }


def create_dataloader(X, y, shuffle=False):
    dataset = TextClassificationDataset(X, y, tokenizer, MAX_LENGTH)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle)

# Create loaders for KATEGORI
train_loader_k = create_dataloader(X_train, yk_train, shuffle=True)
val_loader_k   = create_dataloader(X_val, yk_val)
test_loader_k  = create_dataloader(X_test, yk_test)

# Create loaders for URGENSI
train_loader_u = create_dataloader(X_train, yu_train, shuffle=True)
val_loader_u   = create_dataloader(X_val, yu_val)
test_loader_u  = create_dataloader(X_test, yu_test)

print("Dataloaders created.")
print(f"Kategori - Train: {len(train_loader_k)} batches, Val: {len(val_loader_k)}, Test: {len(test_loader_k)}")
print(f"Urgensi  - Train: {len(train_loader_u)} batches, Val: {len(val_loader_u)}, Test: {len(test_loader_u)}")

In [ ]:
# ============================================================
# 6. CLASS WEIGHTS (handle imbalance)
# ============================================================
def compute_class_weights(labels, num_classes):
    """Compute inverse-frequency class weights."""
    counts = np.bincount(labels, minlength=num_classes)
    weights = len(labels) / (num_classes * counts)
    return torch.tensor(weights, dtype=torch.float).to(device)

num_kategori = len(le_kategori.classes_)
num_urgensi = len(le_urgensi.classes_)

class_weights_k = compute_class_weights(yk_train, num_kategori)
class_weights_u = compute_class_weights(yu_train, num_urgensi)

print("Class weights (kategori):")
for cls, w in zip(le_kategori.classes_, class_weights_k.cpu().numpy()):
    print(f"  {cls}: {w:.4f}")
print("\nClass weights (urgensi):")
for cls, w in zip(le_urgensi.classes_, class_weights_u.cpu().numpy()):
    print(f"  {cls}: {w:.4f}")

In [ ]:
# ============================================================
# 7. MODEL DEFINITION
# ============================================================
def get_model(num_labels):
    """Load IndoBERT with a classification head."""
    config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=num_labels)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config)
    return model.to(device)

model_kategori = get_model(num_kategori)
model_urgensi = get_model(num_urgensi)

print("Models initialized:")
print(f"  Kategori model: {num_kategori} output classes")
print(f"  Urgensi model:  {num_urgensi} output classes")
print(f"  Total params k: {sum(p.numel() for p in model_kategori.parameters()):,}")
print(f"  Total params u: {sum(p.numel() for p in model_urgensi.parameters()):,}")

In [ ]:
# ============================================================
# 8. TRAINING LOOP
# ============================================================
def train_model(model, train_loader, val_loader, class_weights, task_name):
    """Train a single classification model."""
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=0.01
    )
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps,
    )
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    best_f1 = 0.0
    history = []

    for epoch in range(EPOCHS):
        # ---- Training ----
        model.train()
        total_loss = 0
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        # ---- Validation ----
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["label"].to(device)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                preds = torch.argmax(outputs.logits, dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        val_f1 = f1_score(all_labels, all_preds, average="weighted")
        val_acc = accuracy_score(all_labels, all_preds)
        history.append({"epoch": epoch + 1, "loss": avg_loss, "acc": val_acc, "f1": val_f1})

        print(f"[{task_name}] Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

        # Save best model
        if val_f1 > best_f1:
            best_f1 = val_f1
            save_path = OUTPUT_DIR / task_name
            save_path.mkdir(exist_ok=True)
            model.save_pretrained(str(save_path))
            print(f"  -> Saved best {task_name} model (F1={val_f1:.4f})")

    return best_f1, history

print("Both models ready for training.")

In [ ]:
# ============================================================
# 9. TRAIN KATEGORI MODEL
# ============================================================
print("=" * 50)
print("TRAINING: KATEGORI CLASSIFICATION")
print("=" * 50)

best_f1_k, history_k = train_model(
    model_kategori, train_loader_k, val_loader_k,
    class_weights_k, "kategori"
)
print(f"\nBest validation F1 (kategori): {best_f1_k:.4f}")

In [ ]:
# ============================================================
# 10. TRAIN URGENSI MODEL
# ============================================================
print("=" * 50)
print("TRAINING: URGENSI CLASSIFICATION")
print("=" * 50)

best_f1_u, history_u = train_model(
    model_urgensi, train_loader_u, val_loader_u,
    class_weights_u, "urgensi"
)
print(f"\nBest validation F1 (urgensi): {best_f1_u:.4f}")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for name, hist in [("Kategori", history_k), ("Urgensi", history_u)]:
    dfh = pd.DataFrame(hist)
    axes[0].plot(dfh["epoch"], dfh["loss"], marker="o", label=f"{name} Loss")
    axes[1].plot(dfh["epoch"], dfh["f1"], marker="s", label=f"{name} F1")

axes[0].set_title("Training Loss per Epoch")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend(); axes[0].grid(True)
axes[1].set_title("Validation F1 per Epoch")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("F1 (weighted)"); axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_history.png", dpi=150)
plt.show()

In [ ]:
# ============================================================
# 11. EVALUATION ON TEST SET
# ============================================================
def evaluate_model(model, test_loader, label_encoder, task_name):
    """Evaluate model on test set and plot confusion matrix."""
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(outputs.logits, dim=1)
            preds = torch.argmax(outputs.logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    print(f"\n{'='*50}")
    print(f"EVALUATION: {task_name.upper()}")
    print(f"{'='*50}")
    print(f"Accuracy: {accuracy_score(all_labels, all_preds):.4f}")
    print()
    print(classification_report(
        all_labels, all_preds,
        target_names=label_encoder.classes_,
        zero_division=0
    ))

    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=label_encoder.classes_,
        yticklabels=label_encoder.classes_
    )
    plt.title(f"Confusion Matrix - {task_name}")
    plt.ylabel("Actual")
    plt.xlabel("Predicted")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"confusion_matrix_{task_name}.png", dpi=150)
    plt.show()

    report = classification_report(
        all_labels, all_preds,
        target_names=label_encoder.classes_,
        output_dict=True, zero_division=0
    )
    return report, all_preds, all_labels, all_probs

# Load best models
print("Loading best models for evaluation...")
model_k_best = AutoModelForSequenceClassification.from_pretrained(
    str(OUTPUT_DIR / "kategori")
).to(device)
model_u_best = AutoModelForSequenceClassification.from_pretrained(
    str(OUTPUT_DIR / "urgensi")
).to(device)

report_k, preds_k, labels_k, probs_k = evaluate_model(
    model_k_best, test_loader_k, le_kategori, "kategori"
)
report_u, preds_u, labels_u, probs_u = evaluate_model(
    model_u_best, test_loader_u, le_urgensi, "urgensi"
)

In [ ]:
# ============================================================
# 12. ERROR ANALYSIS
# ============================================================
print("=" * 50)
print("ERROR ANALYSIS")
print("=" * 50)

error_idx_k = np.where(preds_k != labels_k)[0]
error_idx_u = np.where(preds_u != labels_u)[0]

print(f"Kategori misclassifications: {len(error_idx_k)} / {len(X_test)}")
print(f"Urgensi misclassifications:  {len(error_idx_u)} / {len(X_test)}")
print()

print("--- Sample Kategori Errors ---")
for i in error_idx_k[:5]:
    print(f"  Text: {X_test[i][:100]}...")
    print(f"  True: {le_kategori.classes_[labels_k[i]]}")
    print(f"  Pred: {le_kategori.classes_[preds_k[i]]}")
    print()

print("--- Sample Urgensi Errors ---")
for i in error_idx_u[:5]:
    print(f"  Text: {X_test[i][:100]}...")
    print(f"  True: {le_urgensi.classes_[labels_u[i]]}")
    print(f"  Pred: {le_urgensi.classes_[preds_u[i]]}")
    print()

In [ ]:
# ============================================================
# 13. SAVE METADATA & ARTIFACTS
# ============================================================
metadata = {
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "num_kategori_classes": num_kategori,
    "num_urgensi_classes": num_urgensi,
    "kategori_classes": le_kategori.classes_.tolist(),
    "urgensi_classes": le_urgensi.classes_.tolist(),
    "best_val_f1_kategori": float(best_f1_k),
    "best_val_f1_urgensi": float(best_f1_u),
    "test_f1_weighted_kategori": float(report_k["weighted avg"]["f1-score"]),
    "test_f1_weighted_urgensi": float(report_u["weighted avg"]["f1-score"]),
    "test_accuracy_kategori": float(report_k["accuracy"]),
    "test_accuracy_urgensi": float(report_u["accuracy"]),
    "train_size": int(len(X_train)),
    "val_size": int(len(X_val)),
    "test_size": int(len(X_test)),
}

with open(OUTPUT_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("=" * 50)
print("METADATA")
print("=" * 50)
print(json.dumps(metadata, indent=2))

In [ ]:
# ============================================================
# 14. FINAL SUMMARY
# ============================================================
print("=" * 60)
print("  TRAINING COMPLETE - MODEL READY FOR DEPLOYMENT")
print("=" * 60)
print(f"\nOutput directory: {OUTPUT_DIR.resolve()}")
print(f"\nGenerated artifacts:")
for f in sorted(Path(OUTPUT_DIR).rglob("*")):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f"  {f.relative_to(OUTPUT_DIR)} ({size_kb:.1f} KB)")

print(f"\n{'='*60}")
print(f"  Test F1 (Kategori): {report_k['weighted avg']['f1-score']:.4f}")
print(f"  Test F1 (Urgensi):  {report_u['weighted avg']['f1-score']:.4f}")
print(f"{'='*60}")

---
### Next Steps

1. **Download the `models/` folder** 
2. **Place it in your application project**
3. **Use `inference_klasifikasi_laporan.ipynb`** to load and test the model

The saved artifacts:
- `kategori/` - IndoBERT fine-tuned for category classification
- `urgensi/` - IndoBERT fine-tuned for urgency classification
- `tokenizer/` - IndoBERT tokenizer files
- `label_encoder_kategori.pkl` - Category label encoder
- `label_encoder_urgensi.pkl` - Urgency label encoder
- `metadata.json` - Model configuration & performance metrics